In [1]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH       = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_INPUT = os.path.join(PATH, 'all_sold_filtered.csv')
LIST_INPUT = os.path.join(PATH, 'all_list_filtered.csv')
DTYPE_SPEC = {'PostalCode': str, 'ListingKey': str}

# =========================================================================
# Helper: section header
# =========================================================================

def section(title):
    print()
    print("=" * 65)
    print(title)
    print("=" * 65)

# =========================================================================
# Load datasets
# =========================================================================

section("Load Datasets")

sold = pd.read_csv(SOLD_INPUT, dtype=DTYPE_SPEC, low_memory=False)
lst  = pd.read_csv(LIST_INPUT, dtype=DTYPE_SPEC, low_memory=False)

print(f"  Sold : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  List : {len(lst):,} rows  |  {lst.shape[1]} columns")

# =========================================================================
# STEP 1 — Parse date columns
# =========================================================================

section("STEP 1 — PARSE DATE COLUMNS")

DATE_COLS = ['CloseDate', 'ListingContractDate',
             'PurchaseContractDate', 'ContractStatusChangeDate']

for df, label in [(sold, 'Sold'), (lst, 'List')]:
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"  [{label}] Date columns parsed.")

# =========================================================================
# STEP 2 — Add YrMo to Listing table
# Sold already has YrMo from Week 6 (derived from CloseDate)
# List needs YrMo derived from ListingContractDate (for New Listings count)
# =========================================================================

section("STEP 2 — ADD YrMo TO LISTING TABLE")

if 'YrMo' not in lst.columns:
    if 'ListingContractDate' in lst.columns:
        lst['YrMo'] = (
            lst['ListingContractDate']
            .dt.to_period('M')
            .astype(str)
        )
        print(f"  [List] YrMo added from ListingContractDate.")
        print(f"  Sample: {lst['YrMo'].dropna().head(3).tolist()}")
    else:
        print("  [List] ListingContractDate not found — YrMo cannot be added.")
else:
    print("  [List] YrMo already exists.")

# Confirm Sold YrMo
if 'YrMo' in sold.columns:
    print(f"  [Sold] YrMo already exists.")
    print(f"  Sample: {sold['YrMo'].dropna().head(3).tolist()}")

# =========================================================================
# STEP 3 — Select only columns needed for Tableau
# =========================================================================

section("STEP 3 — SELECT TABLEAU COLUMNS")

# Sold: columns needed for all 5 dashboards + self-designed dashboard
SOLD_KEEP = [
    # Identifiers
    'ListingKey',

    # Filters (City / County / Zip / PropertySubType)
    'City', 'CountyOrParish', 'PostalCode',
    'PropertyType', 'PropertySubType',

    # Time dimension
    'CloseDate', 'YrMo',

    # Dashboard 1: Monthly Median Close Price
    'ClosePrice',

    # Dashboard 2: Average Days on Market
    'DaysOnMarket',

    # Dashboard 3: Close-to-Original-List Ratio
    'CloseToOriginalListRatio',

    # Dashboard 4: Closed Sales (uses CloseDate count)
    # CloseDate already included above

    # Additional dashboard (Price Per Sqft trend)
    'PricePerSqFt',

    # Extra context
    'ListPrice', 'OriginalListPrice',
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'Latitude', 'Longitude',
    'DistrictName',
    'MLSAreaMajor',
    'ListingToContractDays', 'ContractToCloseDays',
]

# List: columns needed for Dashboard 5 (New Listings)
LIST_KEEP = [
    # Identifiers
    'ListingKey',

    # Filters
    'City', 'CountyOrParish', 'PostalCode',
    'PropertyType', 'PropertySubType',

    # Time dimension
    'ListingContractDate', 'YrMo',

    # Dashboard 5: New Listings (count of ListingKey per YrMo)
    # ListingKey and YrMo already included above

    # Extra context
    'ListPrice', 'LivingArea',
    'DaysOnMarket',
    'MlsStatus',
    'Latitude', 'Longitude',
    'DistrictName',
    'MLSAreaMajor',
]

# Keep only columns that actually exist in the dataframe
sold_cols = [c for c in SOLD_KEEP if c in sold.columns]
list_cols = [c for c in LIST_KEEP  if c in lst.columns]

sold_tableau = sold[sold_cols].copy()
list_tableau = lst[list_cols].copy()

print(f"  Sold: kept {len(sold_cols)} / {len(SOLD_KEEP)} requested columns")
print(f"  List: kept {len(list_cols)} / {len(LIST_KEEP)} requested columns")

missing_sold = [c for c in SOLD_KEEP if c not in sold.columns]
missing_list = [c for c in LIST_KEEP  if c not in lst.columns]
if missing_sold:
    print(f"  ⚠️  Sold missing: {missing_sold}")
if missing_list:
    print(f"  ⚠️  List missing: {missing_list}")

# =========================================================================
# STEP 4 — Filter to Jan 2024 → latest available month
# Handbook: "Analyze data monthly from January 2024 through latest month"
# =========================================================================

section("STEP 4 — FILTER DATE RANGE (Jan 2024 → latest)")

START_PERIOD = '2024-01'

# Sold: filter by YrMo
if 'YrMo' in sold_tableau.columns:
    before = len(sold_tableau)
    sold_tableau = sold_tableau[sold_tableau['YrMo'] >= START_PERIOD]
    print(f"  [Sold] {before:,} → {len(sold_tableau):,} rows "
          f"(from {START_PERIOD})")
    print(f"  [Sold] Date range: "
          f"{sold_tableau['YrMo'].min()} → {sold_tableau['YrMo'].max()}")

# List: filter by YrMo
if 'YrMo' in list_tableau.columns:
    before = len(list_tableau)
    list_tableau = list_tableau[list_tableau['YrMo'] >= START_PERIOD]
    print(f"  [List] {before:,} → {len(list_tableau):,} rows "
          f"(from {START_PERIOD})")
    print(f"  [List] Date range: "
          f"{list_tableau['YrMo'].min()} → {list_tableau['YrMo'].max()}")

# =========================================================================
# STEP 5 — Validate key columns
# =========================================================================

section("STEP 5 — VALIDATION")

print(f"\n  Sold Tableau-ready dataset:")
print(f"    Shape    : {sold_tableau.shape}")
print(f"    Columns  : {list(sold_tableau.columns)}")
print(f"    YrMo range: {sold_tableau['YrMo'].min()} → "
      f"{sold_tableau['YrMo'].max()}")
print(f"    ClosePrice null : {sold_tableau['ClosePrice'].isna().sum():,}")
print(f"    DaysOnMarket null: {sold_tableau['DaysOnMarket'].isna().sum():,}")
if 'CloseToOriginalListRatio' in sold_tableau.columns:
    print(f"    CloseToOriginalListRatio null: "
          f"{sold_tableau['CloseToOriginalListRatio'].isna().sum():,}")

print(f"\n  List Tableau-ready dataset:")
print(f"    Shape    : {list_tableau.shape}")
print(f"    Columns  : {list(list_tableau.columns)}")
print(f"    YrMo range: {list_tableau['YrMo'].min()} → "
      f"{list_tableau['YrMo'].max()}")
print(f"    ListPrice null: {list_tableau['ListPrice'].isna().sum():,}")

# =========================================================================
# STEP 6 — Save Tableau-ready CSVs
# =========================================================================

section("STEP 6 — SAVE TABLEAU-READY FILES")

sold_out = os.path.join(PATH, 'sold_tableau_ready.csv')
list_out = os.path.join(PATH, 'list_tableau_ready.csv')

sold_tableau.to_csv(sold_out, index=False, encoding='utf-8')
list_tableau.to_csv(list_out, index=False, encoding='utf-8')

print(f"  Saved: sold_tableau_ready.csv  "
      f"— {len(sold_tableau):,} rows  |  {sold_tableau.shape[1]} columns")
print(f"  Saved: list_tableau_ready.csv  "
      f"— {len(list_tableau):,} rows  |  {list_tableau.shape[1]} columns")

print(f"\n  These two files are ready to import into Tableau.")
print(f"  Use sold_tableau_ready.csv as the main data source.")
print(f"  Use list_tableau_ready.csv for the New Listings dashboard.")

print("\n" + "=" * 65)
print("WEEK 8 DATA PREPARATION COMPLETE")
print("=" * 65)


Load Datasets
  Sold : 377,473 rows  |  91 columns
  List : 512,749 rows  |  74 columns

STEP 1 — PARSE DATE COLUMNS
  [Sold] Date columns parsed.
  [List] Date columns parsed.

STEP 2 — ADD YrMo TO LISTING TABLE
  [List] YrMo added from ListingContractDate.
  Sample: ['2024-01', '2024-01', '2024-01']
  [Sold] YrMo already exists.
  Sample: ['2024-01', '2024-01', '2024-01']

STEP 3 — SELECT TABLEAU COLUMNS
  Sold: kept 23 / 23 requested columns
  List: kept 16 / 16 requested columns

STEP 4 — FILTER DATE RANGE (Jan 2024 → latest)
  [Sold] 377,473 → 377,473 rows (from 2024-01)
  [Sold] Date range: 2024-01 → 2026-06
  [List] 512,749 → 512,749 rows (from 2024-01)
  [List] Date range: 2024-01 → 2026-06

STEP 5 — VALIDATION

  Sold Tableau-ready dataset:
    Shape    : (377473, 23)
    Columns  : ['ListingKey', 'City', 'CountyOrParish', 'PostalCode', 'PropertyType', 'PropertySubType', 'CloseDate', 'YrMo', 'ClosePrice', 'DaysOnMarket', 'CloseToOriginalListRatio', 'PricePerSqFt', 'ListPrice'